# Hybrid channel simulator
Run cells in order: environment, configuration, fixed positions or orbit, channel generation, and outputs.

`channel_generator.py` selects the backend. `hybrid_model.py` implements the Hybrid model.
The Hybrid model and its fixed 1x1 iso VH RT probe are unchanged; the probe is supplied by `config.py`.

Native RT calls remain in `hybrid_simulator/rt.py`, following the [Sionna RT 1.2.2 Introduction](https://nvlabs.github.io/sionna/v1.2.2/rt/tutorials/Introduction.html). The recorded server version is 1.2.1.

## 0. Environment
Set the expected Mitsuba variant here. This value checks the loaded backend; it does not switch an active backend.
Restart the kernel after updating module files to avoid mixing old and new imports.


In [ ]:
from pathlib import Path
import sys, copy, json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from importlib.metadata import version

WORKING_DIRECTORY = Path.cwd().resolve()
PROJECT = (WORKING_DIRECTORY if (WORKING_DIRECTORY / "config.py").is_file()
           else WORKING_DIRECTORY / "hybrid_simulator")
if not (PROJECT / "config.py").is_file():
    raise RuntimeError("Open the notebook from the repository or hybrid_simulator directory.")
REPOSITORY = PROJECT.parent
sys.path.insert(0, str(REPOSITORY))
from hybrid_simulator import HybridChannel
from hybrid_simulator.config import configure_runtime, notebook_config, notebook_devices, save_run_config
from hybrid_simulator.topology import elevation_deg
from hybrid_simulator.orbit import fetch_tle, sample_positions

configure_runtime()
print("Sionna:", version("sionna"), "RT:", version("sionna-rt"))
# Expected backend; preserve the existing runtime selection.
environment = {"variant": "cuda_ad_mono_polarized"}


## 1. Configuration
Set the mode, frequency, scene, antennas, positions, and solver budget here.
Positions are in meters, orientations in degrees, and antenna spacing in wavelengths.
Enable additional endpoints for fixed-position runs. Orbit runs currently support one satellite and one UE.
The current antenna interface supports iso patterns and V/H/VH polarization; OpenNTN supports regular multi-element arrays using the same row, column, spacing and polarization settings. Output ports use element-major order; native OpenNTN arrays are also retained.


In [ ]:
# Shared parameters and endpoint settings; TLE and timing are set in section B.
settings = {
    ## Simulation
    "mode": "hybrid",                     # "rt", "3gpp", or "hybrid"
    "topology": "fixed",                  # "fixed" (A) or "orbit" (B)
    "directions": ["downlink"],           # "downlink","uplink" 
    "seed": 123,
    "frequency_hz": 2e9,                  # Hz
    "scene_xml": PROJECT.parent / "Scene/NYC_scene/NYC_sionna.xml",
    "output_directory": PROJECT / "results/runs/main_bidirectional",

    ## Native RT solver
    "rt": {
        "max_depth": 3,
        "samples_per_src": {"downlink": 10_000_000, "uplink": 10_000_000},
        "max_num_paths_per_src": 10_000,
        "los": True,
        "specular_reflection": True,
        "diffuse_reflection":True,
        "refraction": True,
        "diffraction": True,
        "edge_diffraction": True,
        "diffraction_lit_region": True,
        "scattering_coefficient": 0.25,
        "power_threshold_db": -25,
    },

    ## Hybrid statistical model
    "hybrid": {
        "backend": "standard",
        "scenario": "NTN-DenseUrban",
        "table_profile": "tr38811_corrected",
        "rays_per_cluster": 20,
        "ray_delay_mode": "kb1",
        "rc_removal_p0": 0.2,
        "rt_xpr_source": "probe_power_ratio",
    },

    ## OpenNTN
    "openntn": {
        "scenario": "dense_urban",
        "los": True,
        "enable_pathloss": True,
        "enable_shadow_fading": True,
        "doppler_enabled": True,
        "num_time_samples": 32,
        "sampling_frequency_hz": 100_000.0,
        "ue_velocity_m_s": [0.0, 0.0, 0.0],
    },

    ## Optional ionosphere effects
    "ionosphere": {
        "group_delay": False,
        "phase_advance": False,
        "absorption": False,
        "scintillation": False,
        "faraday_rotation": False,
        "bending": False,
        "correction": None,
        "faraday": None,
    },

    ## Endpoints
    # Satellite: TX in downlink, RX in uplink. Antennas stay with their endpoints.
    # Spacing: wavelengths. Positions: meters. Orientations: degrees.
    "endpoints": {
        "sat1": {
            "enabled": True,
            "kind": "satellite",
            ## sat1 antenna
            "antenna": {
                "num_rows": 1,
                "num_cols": 1,
                "vertical_spacing": 0.5,
                "horizontal_spacing": 0.5,
                "pattern": "iso",        # Supported pattern: iso
                "polarization": "VH",    # V, H, or VH
            },
            "orientation_deg": [0.0, 0.0, 0.0],  # alpha, beta, gamma
            ## sat1 position
            "position_m": [715052.125, 0.0, 600000.0],
        },
        "ue1": {
            "enabled": True,
            "kind": "ue",
            ## ue1 antenna
            "antenna": {
                "num_rows": 1,
                "num_cols": 1,
                "vertical_spacing": 0.5,
                "horizontal_spacing": 0.5,
                "pattern": "iso",        # Supported pattern: iso
                "polarization": "VH",    # V, H, or VH
            },
            "orientation_deg": [0.0, 0.0, 0.0],  # alpha, beta, gamma
            ## ue1 position
            "position_m": [-1771.537764696344, -1389.7869401296823, 2.0],
        },
        "sat2": {
            "enabled": False,
            "kind": "satellite",
            ## sat2 antenna
            "antenna": {
                "num_rows": 1,
                "num_cols": 1,
                "vertical_spacing": 0.5,
                "horizontal_spacing": 0.5,
                "pattern": "iso",        # Supported pattern: iso
                "polarization": "VH",    # V, H, or VH
            },
            "orientation_deg": [0.0, 0.0, 0.0],  # alpha, beta, gamma
            ## sat2 position
            "position_m": [0.0, 0.0, 600000.0],
        },
        "ue2": {
            "enabled": False,
            "kind": "ue",
            ## ue2 antenna
            "antenna": {
                "num_rows": 1,
                "num_cols": 1,
                "vertical_spacing": 0.5,
                "horizontal_spacing": 0.5,
                "pattern": "iso",        # Supported pattern: iso
                "polarization": "VH",    # V, H, or VH
            },
            "orientation_deg": [0.0, 0.0, 0.0],  # alpha, beta, gamma
            ## ue2 position
            "position_m": [-1761.537764696344, -1389.7869401296823, 2.0],
        },
    },
}

# Convert settings once; internal probe settings are provided by config.py.
config = notebook_config(settings, environment=environment)
devices = notebook_devices(settings)
settings["output_directory"].mkdir(parents=True, exist_ok=True)

## A. Fixed positions
Used when `topology="fixed"`. Every enabled satellite-UE pair is evaluated for the selected directions.


In [ ]:
if settings["topology"] == "fixed":
    times = np.array([0.0])
    dates = [None]
    positions = {
        name: np.asarray([device["position"]], dtype=float)
        for name, device in devices.items()
    }
    satellite_names = [name for name in devices if settings["endpoints"][name]["kind"] == "satellite"]
    ue_names = [name for name in devices if settings["endpoints"][name]["kind"] == "ue"]
    links = {}
    for sat_name in satellite_names:
        for ue_name in ue_names:
            links[f"{sat_name}_{ue_name}"] = {"satellite": sat_name, "ue": ue_name}
    print("Links to simulate:", list(links))

## B. TLE orbit
Used when `topology="orbit"`. Set the TLE and time interval below.
Hybrid retains a separate fixed-scatterer state for each direction. Sparse geometry snapshots do not resolve carrier-scale fast fading.


In [ ]:
if settings['topology'] == "orbit":
    from datetime import datetime, timezone

    if settings['mode'] == "3gpp":
        raise ValueError("軌道段目前支援 RT/Hybrid；OpenNTN 的軌道狀態尚未接入。")
    satellite_names = [name for name in devices if settings["endpoints"][name]["kind"] == "satellite"]
    ue_names = [name for name in devices if settings["endpoints"][name]["kind"] == "ue"]
    if len(satellite_names) != 1 or len(ue_names) != 1:
        raise ValueError("軌道入口目前限一顆衛星與一個 UE，請在 configuration 只啟用這兩個端點。")
    sat_name, ue_name = satellite_names[0], ue_names[0]

    ## Orbit: NORAD, TLE, and timing
    NORAD_ID = 25544
    START_UTC = datetime.now(timezone.utc).isoformat()
    DURATION_S = 4
    INTERVAL_S = 1
    ORIGIN_LON_LAT_HEIGHT = [-74.006, 40.7128, 0.0]  # Example origin; verify the city georeference before experiments.
    TLE_FILE = None  # Set a local TLE file to avoid downloading one.
    if TLE_FILE is None:
        TLE_FILE = fetch_tle(NORAD_ID)
    config["coordinates"]["origin_geodetic_deg_m"] = ORIGIN_LON_LAT_HEIGHT
    # Antennas and UE position come from configuration; TLE determines satellite positions.
    config["devices"] = {
        "satellite": copy.deepcopy(devices[sat_name]),
        "ue": copy.deepcopy(devices[ue_name]),
    }
    trajectory_config = copy.deepcopy(config)
    trajectory_config["trajectory"] = {
        "model": (
            "fixed_scatterer_segment" if settings['mode'] == "hybrid" else "independent_snapshots"
        ),
        "duration_s": DURATION_S,
        "interval_s": INTERVAL_S,
        "start_utc": START_UTC,
        "orbit": {"kind": "tle", "tle_file": str(TLE_FILE), "max_epoch_age_days": 3.0},
    }
    times, positions, velocities, dates, provenance = sample_positions(
        trajectory_config
    )
    elevations = np.array(
        [
            elevation_deg({name: p[i] for name, p in positions.items()})
            for i in range(len(times))
        ]
    )
    if np.any((elevations < 10.0) | (elevations > 90.0)):
        raise ValueError("不在 NTN 表的 10–90 度範圍，請選衛星可見時段。")
    if settings['mode'] == "hybrid" and np.max(abs(elevations - elevations[0])) > 1.0:
        raise ValueError("超過既有短段的 1 度仰角範圍。")
    config["devices"]["satellite"]["position"] = positions["satellite"][0].tolist()
    provenance.update(start_utc=START_UTC, utc=dates, norad_id=NORAD_ID)
    np.savez_compressed(
        settings['output_directory'] / "orbit_positions.npz",
        time_s=times,
        satellite_velocity_m_s=velocities["satellite"],
        ue_velocity_m_s=velocities["ue"],
        **positions,
    )
    (settings['output_directory'] / "orbit_source.json").write_text(json.dumps(provenance, indent=2))

    # Orbit currently supports one satellite and one UE; reuse fixed-mode link names.
    positions = {sat_name: positions["satellite"], ue_name: positions["ue"]}
    links = {f"{sat_name}_{ue_name}": {"satellite": sat_name, "ue": ue_name}}

## 2. Save configuration and create channel generators
Each link and direction has its own scene and temporal state.
- RT: native Sionna paths and existing postprocessing.
- Hybrid: RT results followed by `hybrid_model.py`.
- 3GPP: native OpenNTN through `hybrid_simulator/openntn.py`.


In [ ]:
# Save the configurations used below, with positions from section A or B.
CONFIG_FILE = settings["output_directory"] / "config.json"
configs = save_run_config(
    CONFIG_FILE, settings, config, links=links, positions=positions,
    trajectory_config=trajectory_config if settings["topology"] == "orbit" else None,
)
print("Run configuration:", CONFIG_FILE)
if settings["mode"] == "3gpp":
    # Run OpenNTN on TensorFlow CPU; the RT GPU backend is unchanged.
    import tensorflow as tf
    tf.config.set_visible_devices([], "GPU")

# Each link and direction owns its scene and temporal state.
channels = {}
results = {link: {direction: [] for direction in settings['directions']} for link in links}
summary = []
for link in links:
    channels[link] = {}
    for direction in settings['directions']:
        direction_config = configs[link][direction]
        channels[link][direction] = HybridChannel(
            direction_config,
            device_names=links[link],
            persistent=(settings['topology'] == "orbit" and settings['mode'] == "hybrid"),
        )

## 3. Generate and save channels
Each call uses the current endpoint positions. RT/Hybrid trace each direction separately.
Native RT coefficients, existing thresholds, delay references, and Hybrid generation rules are unchanged.
Each snapshot saves CSV, NPZ, and JSON files. Reusing an output directory overwrites matching files.


In [ ]:
for i, seconds in enumerate(times):
    for link, names in links.items():
        link_positions = {role: positions[name][i] for role, name in names.items()}
        for direction in settings['directions']:
            channel = channels[link][direction]
            result = channel(
                satellite_position_m=link_positions["satellite"],
                ue_position_m=link_positions["ue"],
                time_s=float(seconds),
            )
            result.metadata.update(
                time_s=float(seconds), time_utc=dates[i],
                link_name=link, endpoint_names=names.copy(),
            )
            label = "fixed" if settings['topology'] == "fixed" else f"orbit_{i:04d}"
            snapshot_directory = settings['output_directory'] / link / direction / label
            result.save(snapshot_directory)
            result.runtime_context["output_directory"] = str(snapshot_directory.resolve())
            print("Saved CSV / NPZ / JSON:", snapshot_directory.resolve())
            results[link][direction].append(result)
            rt_info = result.metadata.get("rt", {})
            summary.append({
                "link": link,
                "time (s)": float(seconds),
                "direction": direction,
                "native RT paths": rt_info.get("native_valid_path_count"),
                "RT paths >= -25 dB": rt_info.get("retained_rt_path_count"),
                "output rays/taps": len(result.tau),
            })
            print(link, direction, dates[i], "output rays/taps:", len(result.tau))
display(pd.DataFrame(summary))

## 4. Inspect outputs
Inspect the last snapshot of each link and direction. SAMPLE_INDEX selects the within-snapshot time sample.
The tables show source counts, per-port power gain, phase and power-weighted delay statistics. Gain is in dB, not dBm. Ports include polarization.
inspection_summary.csv, port_metrics.csv and source_summary.csv are refreshed for the selected sample. Original channel CSV/NPZ files retain all samples.
The per-ray PDP below selects one port pair and normalizes individual ray powers. SHOW_DEBUG_SHAPES enables array dimensions.


In [ ]:
# Inspect the last snapshot of each link; select a within-snapshot time sample.
SAMPLE_INDEX = 0
SHOW_DEBUG_SHAPES = False

def power_db(power):
    """Dimensionless channel power gain in dB, not received power in dBm."""
    return 10 * np.log10(power) if power > 0 else -np.inf

for link in links:
    for direction in settings['directions']:
        snapshots = results[link][direction]
        if not snapshots:
            raise RuntimeError("Generate the channel before inspecting outputs.")
        result = snapshots[-1]
        a, tau, h = result.a, result.tau, result.h
        if not 0 <= SAMPLE_INDEX < a.shape[-1]:
            raise ValueError(f"SAMPLE_INDEX must be between 0 and {a.shape[-1] - 1}.")
        sample_a = a[..., SAMPLE_INDEX]
        sample_h = h[..., SAMPLE_INDEX]
        sample_offsets = result.extra_arrays.get("time_s", np.zeros(a.shape[-1]))
        metadata = result.metadata
        rt_info = metadata.get("rt", {})
        print(f"\n{link} / {direction}: last snapshot, sample {SAMPLE_INDEX}")
        overview = {
            "link": link, "direction": direction,
            "mode": metadata.get("mode", "unavailable"),
            "frequency_GHz": metadata.get("frequency_hz", np.nan) / 1e9,
            "snapshot_UTC": metadata.get("time_utc", "unavailable"),
            "snapshot_time_s": metadata.get("time_s", np.nan),
            "sample_index": SAMPLE_INDEX,
            "sample_offset_s": float(sample_offsets[SAMPLE_INDEX]),
            "RX_ports": a.shape[0], "TX_ports": a.shape[1],
            "time_samples": a.shape[-1], "output_rays_or_taps": len(tau),
            "native_valid_RT_paths": rt_info.get("native_valid_path_count"),
            "retained_RT_paths": rt_info.get("retained_rt_path_count"),
            "delay_reference_ms": result.delay_reference_s * 1e3,
        }
        display(pd.DataFrame(overview.items(), columns=["quantity", "value"]))
        print("Ports include polarization: one VH element contributes two ports.")

        source_rows = []
        all_port_power = np.sum(np.abs(sample_a) ** 2, axis=(0, 1))
        for source in np.unique(result.sources):
            selected = result.sources == source
            source_rows.append({
                "source": str(source), "output_rays_or_taps": int(selected.sum()),
                "nonzero_rays_or_taps": int(np.count_nonzero(all_port_power[selected] > 0)),
                "summed_port_power_gain_dB": power_db(float(all_port_power[selected].sum())),
            })
        print("Output composition (RT-derived rays are not native geometric path counts):")
        source_table = pd.DataFrame(source_rows)
        display(source_table)

        port_rows = []
        for rx_port in range(a.shape[0]):
            for tx_port in range(a.shape[1]):
                powers = np.abs(sample_a[rx_port, tx_port]) ** 2
                valid = np.isfinite(tau) & (tau >= 0) & np.isfinite(powers) & (powers > 0)
                total_power = float(powers.sum())
                mean_delay = rms_delay = span = np.nan
                if np.any(valid):
                    delays_ns = tau[valid] * 1e9
                    weights = powers[valid] / powers[valid].sum()
                    mean_delay = float(np.sum(weights * delays_ns))
                    rms_delay = float(np.sqrt(np.sum(weights * (delays_ns - mean_delay) ** 2)))
                    span = float(np.ptp(delays_ns))
                coefficient = sample_h[rx_port, tx_port]
                port_rows.append({
                    "sample_index": SAMPLE_INDEX, "RX_port": rx_port, "TX_port": tx_port,
                    "nonzero_rays_or_taps": int(valid.sum()),
                    "distinct_excess_delays": len(np.unique(tau[valid])),
                    "sum_path_power_gain_dB": power_db(total_power),
                    "coherent_H_gain_dB": power_db(float(abs(coefficient) ** 2)),
                    "H_phase_deg": float(np.angle(coefficient, deg=True)) if abs(coefficient) > 0 else np.nan,
                    "mean_excess_delay_ns": mean_delay,
                    "RMS_delay_spread_ns": rms_delay,
                    "excess_delay_span_ns": span,
                })
        port_table = pd.DataFrame(port_rows)
        print("Per-port channel metrics (indices start at zero):")
        display(port_table)
        print("sum_path_power: sum |a|^2; coherent_H: |sum a|^2, including phase interference.")
        print("Gains are dB, not dBm. Undefined delays/phases are NaN; zero gain is -inf.")
        print("RMS delay spread uses nonzero paths with valid delays; no extra power threshold.")

        label = "fixed" if settings['topology'] == "fixed" else f"orbit_{len(snapshots)-1:04d}"
        output_directory = Path(result.runtime_context.get(
            "output_directory", settings['output_directory'] / link / direction / label
        )).resolve()
        if (output_directory / "channel.npz").exists():
            pd.DataFrame([overview]).to_csv(output_directory / "inspection_summary.csv", index=False, encoding="utf-8-sig")
            port_table.to_csv(output_directory / "port_metrics.csv", index=False, encoding="utf-8-sig")
            source_table.to_csv(output_directory / "source_summary.csv", index=False, encoding="utf-8-sig")
            print("CSV directory:", output_directory)
            display(pd.DataFrame([
                {"file": name, "contents": meaning, "exists": (output_directory / name).exists()}
                for name, meaning in [
                    ("inspection_summary.csv", "Snapshot and channel dimensions"),
                    ("port_metrics.csv", "Power, phase and delay statistics per RX/TX port"),
                    ("source_summary.csv", "RT / RC / OpenNTN output composition"),
                    ("channel_paths.csv", "Complex coefficient and delay of every path"),
                    ("channel_matrix.csv", "Coherent H for all time samples and port pairs"),
                    ("pdp.csv", "Path powers grouped by exactly equal delay, summed over ports"),
                    ("cfr.csv", "Frequency response for all samples and port pairs"),
                ]
            ]))
        else:
            print("No saved snapshot found at:", output_directory)
            print("Inspection CSVs were not written; save this result first.")
        print("The per-ray PDP below has independent snapshot, sample and port selections.")
        if SHOW_DEBUG_SHAPES:
            print("a [RX, TX, ray/tap, time]:", a.shape)
            print("tau [ray/tap]:", tau.shape)
            print("h [RX, TX, time]:", h.shape)


## 5. Custom postprocessing
Per-ray PDP matches the GitHub plotting convention: select one RX/TX port pair, plot each nonzero ray separately, and normalize each panel to its own strongest ray (0 dB). Equal-delay rays overlap; no coherent or incoherent grouping is applied.
The default is snapshot 0, time sample 0. Set SNAPSHOT_INDEX=-1 for the latest snapshot. For MIMO, change RX_PORT and TX_PORT; VH elements have two ports. This plot does not show absolute gain differences between port pairs.
A separate ray_pdp_*.csv matches the plotted points. The existing pdp.csv retains its all-port, unnormalized incoherent power definition.


In [ ]:
# Match the GitHub plot: one port pair, individual rays, peak-normalized power.
RX_PORT = 0
TX_PORT = 0
SNAPSHOT_INDEX = 0  # First generated snapshot; use -1 for the last snapshot.
TIME_SAMPLE = 0

for link in links:
    fig, axes = plt.subplots(
        1, len(settings["directions"]),
        figsize=(6 * len(settings["directions"]), 3), sharey=True, squeeze=False,
    )
    plot_rows = []
    for ax, direction in zip(axes[0], settings["directions"]):
        snapshots = results[link][direction]
        if not snapshots:
            raise RuntimeError("Generate the channel before plotting.")
        result = snapshots[SNAPSHOT_INDEX]
        a = result.a
        if not (0 <= RX_PORT < a.shape[0] and 0 <= TX_PORT < a.shape[1]):
            raise ValueError(f"{link}/{direction}: RX_PORT must be 0..{a.shape[0]-1}; TX_PORT must be 0..{a.shape[1]-1}.")
        if not 0 <= TIME_SAMPLE < a.shape[-1]:
            raise ValueError(f"TIME_SAMPLE must be 0..{a.shape[-1]-1}.")
        power = np.abs(a[RX_PORT, TX_PORT, :, TIME_SAMPLE]) ** 2
        active = (power > 0) & np.isfinite(power) & np.isfinite(result.tau)
        if np.any(active):
            delays = result.tau[active]
            powers = power[active].astype(float)
            excess_ns = (delays - delays.min()) * 1e9
            relative_power_db = 10 * np.log10(powers / powers.max())
            ax.stem(excess_ns, relative_power_db,
                    bottom=float(relative_power_db.min()) - 5, basefmt=" ")
            for path_index, delay, gain in zip(np.flatnonzero(active), excess_ns, relative_power_db):
                plot_rows.append({
                    "direction": direction,
                    "snapshot_index": SNAPSHOT_INDEX % len(snapshots),
                    "time_sample": TIME_SAMPLE,
                    "rx_port": RX_PORT, "tx_port": TX_PORT,
                    "path_index": int(path_index),
                    "excess_delay_ns": float(delay),
                    "relative_ray_power_db": float(gain),
                })
        else:
            ax.text(0.5, 0.5, "No nonzero rays", transform=ax.transAxes, ha="center")
        ax.set(
            xlabel="Excess delay (ns)", ylabel="Relative ray power (dB)",
            title=f"{link} / {direction} / RX{RX_PORT}-TX{TX_PORT}\n"
                  f"Snapshot {SNAPSHOT_INDEX % len(snapshots)}, sample {TIME_SAMPLE}",
        )
        ax.grid(alpha=0.3)
    fig.tight_layout()
    figure_directory = Path(settings['output_directory']) / link
    figure_directory.mkdir(parents=True, exist_ok=True)
    figure_stem = f"ray_pdp_rx{RX_PORT}_tx{TX_PORT}_snapshot{SNAPSHOT_INDEX}_sample{TIME_SAMPLE}"
    figure_file = figure_directory / (figure_stem + ".png")
    fig.savefig(figure_file, dpi=160)
    pd.DataFrame(plot_rows, columns=[
        "direction", "snapshot_index", "time_sample", "rx_port", "tx_port",
        "path_index", "excess_delay_ns", "relative_ray_power_db",
    ]).to_csv(figure_directory / (figure_stem + ".csv"), index=False, encoding="utf-8-sig")
    print("Saved per-ray PDP and matching CSV:", figure_file.resolve())
    print("Each panel has its own 0 dB peak; equal-delay rays overlap without summation.")
    print("This plot differs from pdp.csv, which sums unnormalized path powers across ports.")
    plt.show()


In [ ]:
SHOW_SCENE = True   #True, False
if SHOW_SCENE and settings['mode'] != "3gpp":
    from sionna.rt import Camera
    for link in links:
        for direction in settings['directions']:
            result = results[link][direction][-1]
            native_scene = result.runtime_context["scene"]
            native_paths = result.runtime_context["paths"]
            ue = np.asarray(result.metadata["positions_scene_m"]["ue"])
            camera = Camera(
                position=(ue + [25.0, -25.0, 30.0]).tolist(),
                look_at=(ue).tolist(),
            )
            print(link, direction, "Native RT paths, including paths below the power threshold")
            native_scene.render(
                camera=camera, paths=native_paths, resolution=[900, 600], num_samples=128
            )